# Solar Irradiance Data Visualization Analysis

This notebook analyzes and visualizes DHI (Diffuse Horizontal Irradiance), DNI (Direct Normal Irradiance), CDHI (Clear Sky DHI), and CDNI (Clear Sky DNI) data over the year to identify seasonal patterns and trends.

## Objective
Create comprehensive visualizations to make seasonal conclusions about solar irradiance data visually digestible and identify key patterns in the data.

## 1. Import Required Libraries
Import pandas, matplotlib, seaborn, numpy, and datetime libraries for data manipulation and visualization.

In [ ]:
# Import Required Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## 2. Load and Explore the Dataset
Load the BigData.csv file, examine its structure, check for missing values, and display basic statistics for DHI, DNI, CDHI, CDNI columns.

In [ ]:
# Load the dataset
df = pd.read_csv('BigData.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# Check for DHI, DNI, CDHI, CDNI columns and their basic statistics
irradiance_columns = ['DHI', 'DNI', 'CDHI', 'CDNI']
available_columns = [col for col in irradiance_columns if col in df.columns]

print("Available irradiance columns:", available_columns)
print("\nBasic Statistics for Irradiance Data:")
if available_columns:
    print(df[available_columns].describe())
    
    print("\nMissing Values:")
    print(df[available_columns].isnull().sum())
    
    print("\nData Types:")
    print(df[available_columns].dtypes)
else:
    print("Irradiance columns not found. Available columns:")
    print(df.columns.tolist())

## 3. Data Preprocessing and Date Handling
Parse date/time columns, create datetime index, handle any missing or invalid data, and prepare the dataset for time series analysis.

In [ ]:
# Identify date/time columns
date_columns = []
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['date', 'time', 'timestamp', 'year', 'month', 'day']):
        date_columns.append(col)

print("Potential date/time columns:", date_columns)
print("\nSample values from potential date columns:")
for col in date_columns[:3]:  # Show first 3 date columns
    print(f"{col}: {df[col].head().tolist()}")

In [ ]:
# Create datetime index based on available date/time information
def create_datetime_index(df):
    # Check if there's a direct datetime column
    if 'Timestamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['Timestamp'])
    elif 'Date' in df.columns:
        if 'Time' in df.columns:
            df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
        else:
            df['datetime'] = pd.to_datetime(df['Date'])
    # Try to combine year, month, day, hour columns
    elif all(col in df.columns for col in ['Year', 'Month', 'Day']):
        if 'Hour' in df.columns:
            df['datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour']])
        else:
            df['datetime'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    else:
        # Create a simple date range if no date columns found
        print("No clear date columns found. Creating artificial datetime index...")
        df['datetime'] = pd.date_range(start='2023-01-01', periods=len(df), freq='H')
    
    return df

# Apply datetime creation
df = create_datetime_index(df)
df.set_index('datetime', inplace=True)

print("Datetime index created successfully!")
print("Date range:", df.index.min(), "to", df.index.max())
print("Data frequency:", df.index.freq if hasattr(df.index, 'freq') else "Variable")

In [ ]:
# Handle missing and invalid data
print("Handling missing and invalid data...")

# Remove negative values (physically impossible for irradiance)
for col in available_columns:
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            print(f"Removing {negative_count} negative values from {col}")
            df[col] = df[col].where(df[col] >= 0, np.nan)

# Handle missing values
print("\nMissing values after cleaning:")
missing_data = df[available_columns].isnull().sum()
print(missing_data)

# Fill missing values with interpolation for time series data
for col in available_columns:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].interpolate(method='time', limit_direction='both')

print("\nMissing values after interpolation:")
print(df[available_columns].isnull().sum())

# Add time-based features for analysis
df['month'] = df.index.month
df['day_of_year'] = df.index.dayofyear
df['hour'] = df.index.hour if hasattr(df.index, 'hour') else 12  # Default to noon if no hour info

## 4. Create Time Series Plots
Generate line plots showing DHI, DNI, CDHI, CDNI values over time, including daily, monthly, and seasonal aggregations.

In [ ]:
# Create comprehensive time series plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Solar Irradiance Time Series Analysis', fontsize=16, fontweight='bold')

# Plot 1: All irradiance components over time (full dataset)
ax1 = axes[0, 0]
for col in available_columns:
    if col in df.columns:
        # Sample data if too many points for visibility
        if len(df) > 10000:
            sample_df = df.sample(n=10000).sort_index()
        else:
            sample_df = df
        ax1.plot(sample_df.index, sample_df[col], label=col, alpha=0.7, linewidth=1)

ax1.set_title('All Irradiance Components Over Time')
ax1.set_xlabel('Date')
ax1.set_ylabel('Irradiance (W/m²)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Monthly averages
ax2 = axes[0, 1]
monthly_avg = df[available_columns].groupby(df['month']).mean()
for col in available_columns:
    if col in monthly_avg.columns:
        ax2.plot(monthly_avg.index, monthly_avg[col], marker='o', linewidth=2, label=col)

ax2.set_title('Monthly Average Irradiance')
ax2.set_xlabel('Month')
ax2.set_ylabel('Average Irradiance (W/m²)')
ax2.set_xticks(range(1, 13))
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Daily pattern (hourly averages)
ax3 = axes[1, 0]
if 'hour' in df.columns and df['hour'].nunique() > 1:
    hourly_avg = df[available_columns].groupby(df['hour']).mean()
    for col in available_columns:
        if col in hourly_avg.columns:
            ax3.plot(hourly_avg.index, hourly_avg[col], marker='o', linewidth=2, label=col)
    ax3.set_title('Daily Pattern - Hourly Averages')
    ax3.set_xlabel('Hour of Day')
    ax3.set_xticks(range(0, 24, 2))
else:
    ax3.text(0.5, 0.5, 'Hourly data not available', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Daily Pattern - Not Available')

ax3.set_ylabel('Average Irradiance (W/m²)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Seasonal pattern (day of year)
ax4 = axes[1, 1]
seasonal_avg = df[available_columns].groupby(df['day_of_year']).mean()
for col in available_columns:
    if col in seasonal_avg.columns:
        ax4.plot(seasonal_avg.index, seasonal_avg[col], alpha=0.8, linewidth=1.5, label=col)

ax4.set_title('Seasonal Pattern - Day of Year Averages')
ax4.set_xlabel('Day of Year')
ax4.set_ylabel('Average Irradiance (W/m²)')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Add seasonal markers
season_starts = [1, 80, 172, 266]  # Approximate start of seasons
season_names = ['Winter', 'Spring', 'Summer', 'Fall']
for start, name in zip(season_starts, season_names):
    ax4.axvline(x=start, color='gray', linestyle='--', alpha=0.5)
    ax4.text(start, ax4.get_ylim()[1] * 0.9, name, rotation=90, alpha=0.7)

plt.tight_layout()
plt.show()

## 5. Generate Comparative Visualizations
Create subplots, overlapping charts, and heatmaps to compare the four irradiance metrics and identify patterns and correlations.

In [ ]:
# Create comparative visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Comparative Analysis of Solar Irradiance Components', fontsize=16, fontweight='bold')

# Plot 1: Box plots by month
ax1 = axes[0, 0]
monthly_data = []
months = []
colors = []
for month in range(1, 13):
    month_data = df[df['month'] == month]
    for col in available_columns:
        if col in month_data.columns and not month_data[col].empty:
            monthly_data.append(month_data[col].dropna())
            months.append(f'{month}\n{col}')
            colors.append(available_columns.index(col))

if monthly_data:
    bp = ax1.boxplot(monthly_data, labels=months, patch_artist=True)
    # Color the boxes
    for patch, color_idx in zip(bp['boxes'], colors):
        patch.set_facecolor(plt.cm.Set3(color_idx / len(available_columns)))
    
    ax1.set_title('Monthly Distribution Comparison')
    ax1.set_ylabel('Irradiance (W/m²)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)

# Plot 2: Correlation heatmap
ax2 = axes[0, 1]
if len(available_columns) >= 2:
    correlation_matrix = df[available_columns].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, ax=ax2, cbar_kws={'shrink': 0.8})
    ax2.set_title('Correlation Matrix - Irradiance Components')
else:
    ax2.text(0.5, 0.5, 'Insufficient data for correlation', ha='center', va='center', transform=ax2.transAxes)

# Plot 3: Scatter plot matrix (for first two available columns)
ax3 = axes[1, 0]
if len(available_columns) >= 2:
    col1, col2 = available_columns[0], available_columns[1]
    scatter_data = df[[col1, col2]].dropna()
    if len(scatter_data) > 10000:
        scatter_data = scatter_data.sample(n=10000)
    
    ax3.scatter(scatter_data[col1], scatter_data[col2], alpha=0.5, s=1)
    ax3.set_xlabel(f'{col1} (W/m²)')
    ax3.set_ylabel(f'{col2} (W/m²)')
    ax3.set_title(f'{col1} vs {col2}')
    ax3.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(scatter_data[col1], scatter_data[col2], 1)
    p = np.poly1d(z)
    ax3.plot(scatter_data[col1].sort_values(), p(scatter_data[col1].sort_values()), "r--", alpha=0.8)

# Plot 4: Monthly heatmap
ax4 = axes[1, 1]
if len(available_columns) > 0:
    # Create a pivot table for heatmap
    monthly_summary = df.groupby('month')[available_columns].mean()
    sns.heatmap(monthly_summary.T, annot=True, fmt='.1f', cmap='YlOrRd',
                ax=ax4, cbar_kws={'shrink': 0.8})
    ax4.set_title('Monthly Average Irradiance Heatmap')
    ax4.set_xlabel('Month')
    ax4.set_ylabel('Irradiance Type')

plt.tight_layout()
plt.show()

In [ ]:
# Additional comparative visualization: Direct vs Clear Sky comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Direct vs Clear Sky Irradiance Comparison', fontsize=16, fontweight='bold')

# DHI vs CDHI comparison
ax1 = axes[0]
if 'DHI' in df.columns and 'CDHI' in df.columns:
    monthly_dhi = df.groupby('month')['DHI'].mean()
    monthly_cdhi = df.groupby('month')['CDHI'].mean()
    
    x = range(1, 13)
    width = 0.35
    
    bars1 = ax1.bar([i - width/2 for i in x], monthly_dhi, width, label='DHI (Actual)', alpha=0.8)
    bars2 = ax1.bar([i + width/2 for i in x], monthly_cdhi, width, label='CDHI (Clear Sky)', alpha=0.8)
    
    ax1.set_title('DHI vs CDHI - Monthly Comparison')
    ax1.set_xlabel('Month')
    ax1.set_ylabel('Irradiance (W/m²)')
    ax1.set_xticks(x)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# DNI vs CDNI comparison
ax2 = axes[1]
if 'DNI' in df.columns and 'CDNI' in df.columns:
    monthly_dni = df.groupby('month')['DNI'].mean()
    monthly_cdni = df.groupby('month')['CDNI'].mean()
    
    bars1 = ax2.bar([i - width/2 for i in x], monthly_dni, width, label='DNI (Actual)', alpha=0.8)
    bars2 = ax2.bar([i + width/2 for i in x], monthly_cdni, width, label='CDNI (Clear Sky)', alpha=0.8)
    
    ax2.set_title('DNI vs CDNI - Monthly Comparison')
    ax2.set_xlabel('Month')
    ax2.set_ylabel('Irradiance (W/m²)')
    ax2.set_xticks(x)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Add Statistical Analysis and Trends
Calculate moving averages, identify peak periods, perform correlation analysis, and add trend lines to support visual conclusions.

In [ ]:
# Statistical analysis and trends
print("=== STATISTICAL ANALYSIS SUMMARY ===\n")

# Calculate basic statistics
for col in available_columns:
    if col in df.columns:
        print(f"{col} Statistics:")
        print(f"  Mean: {df[col].mean():.2f} W/m²")
        print(f"  Std:  {df[col].std():.2f} W/m²")
        print(f"  Max:  {df[col].max():.2f} W/m² (Date: {df[col].idxmax()})")
        print(f"  Min:  {df[col].min():.2f} W/m² (Date: {df[col].idxmin()})")
        print()

# Identify peak periods
print("=== PEAK PERIODS ANALYSIS ===")
monthly_avg = df[available_columns].groupby(df['month']).mean()
for col in available_columns:
    if col in monthly_avg.columns:
        peak_month = monthly_avg[col].idxmax()
        peak_value = monthly_avg[col].max()
        low_month = monthly_avg[col].idxmin()
        low_value = monthly_avg[col].min()
        
        month_names = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        
        print(f"{col}:")
        print(f"  Peak: {month_names[peak_month]} ({peak_value:.2f} W/m²)")
        print(f"  Low:  {month_names[low_month]} ({low_value:.2f} W/m²)")
        print(f"  Seasonal variation: {((peak_value - low_value) / low_value * 100):.1f}%")
        print()

In [ ]:
# Moving averages and trend analysis
window_size = min(30, len(df) // 10)  # 30-day window or 10% of data, whichever is smaller

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Trend Analysis with Moving Averages', fontsize=16, fontweight='bold')

for idx, col in enumerate(available_columns[:4]):  # Limit to first 4 columns
    ax = axes[idx // 2, idx % 2]
    
    if col in df.columns:
        # Original data (sampled if too large)
        if len(df) > 5000:
            sample_df = df.sample(n=5000).sort_index()
        else:
            sample_df = df
            
        ax.plot(sample_df.index, sample_df[col], alpha=0.3, color='lightblue', 
                label=f'{col} (Raw)', linewidth=0.5)
        
        # Moving average
        moving_avg = df[col].rolling(window=window_size, center=True).mean()
        ax.plot(moving_avg.index, moving_avg, color='red', linewidth=2, 
                label=f'{col} ({window_size}-period MA)')
        
        # Trend line (linear regression)
        if len(df) > 1:
            x_numeric = np.arange(len(df))
            valid_mask = ~df[col].isna()
            if valid_mask.sum() > 1:
                z = np.polyfit(x_numeric[valid_mask], df[col][valid_mask], 1)
                p = np.poly1d(z)
                trend_line = p(x_numeric)
                ax.plot(df.index, trend_line, color='green', linewidth=2, 
                        linestyle='--', label=f'{col} (Trend)')
                
                # Calculate trend direction
                trend_slope = z[0] * len(df)  # Total change over dataset
                trend_direction = "↗ Increasing" if trend_slope > 0 else "↘ Decreasing" if trend_slope < 0 else "→ Stable"
                ax.text(0.02, 0.98, f'Trend: {trend_direction}', transform=ax.transAxes, 
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax.set_title(f'{col} - Trend Analysis')
        ax.set_ylabel('Irradiance (W/m²)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{col} data not available', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{col} - Not Available')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis and seasonal patterns
print("=== CORRELATION ANALYSIS ===")
if len(available_columns) >= 2:
    correlation_matrix = df[available_columns].corr()
    print("Correlation Matrix:")
    print(correlation_matrix.round(3))
    print()
    
    # Find strongest correlations
    correlations = []
    for i in range(len(available_columns)):
        for j in range(i+1, len(available_columns)):
            col1, col2 = available_columns[i], available_columns[j]
            if col1 in correlation_matrix.columns and col2 in correlation_matrix.columns:
                corr_value = correlation_matrix.loc[col1, col2]
                correlations.append((col1, col2, corr_value))
    
    correlations.sort(key=lambda x: abs(x[2]), reverse=True)
    
    print("Strongest correlations:")
    for col1, col2, corr in correlations:
        strength = "Very Strong" if abs(corr) > 0.8 else "Strong" if abs(corr) > 0.6 else "Moderate" if abs(corr) > 0.4 else "Weak"
        direction = "Positive" if corr > 0 else "Negative"
        print(f"  {col1} - {col2}: {corr:.3f} ({strength} {direction})")

print("\n=== SEASONAL PATTERNS SUMMARY ===")

# Define seasons
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)
seasonal_avg = df.groupby('season')[available_columns].mean()

print("Seasonal Averages:")
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
for season in season_order:
    if season in seasonal_avg.index:
        print(f"\n{season}:")
        for col in available_columns:
            if col in seasonal_avg.columns:
                value = seasonal_avg.loc[season, col]
                print(f"  {col}: {value:.2f} W/m²")

# Calculate seasonal variation
print("\n=== SEASONAL VARIATION ANALYSIS ===")
for col in available_columns:
    if col in seasonal_avg.columns:
        max_season = seasonal_avg[col].idxmax()
        min_season = seasonal_avg[col].idxmin()
        max_value = seasonal_avg[col].max()
        min_value = seasonal_avg[col].min()
        variation = ((max_value - min_value) / min_value) * 100
        
        print(f"{col}:")
        print(f"  Peak season: {max_season} ({max_value:.2f} W/m²)")
        print(f"  Low season: {min_season} ({min_value:.2f} W/m²)")
        print(f"  Seasonal variation: {variation:.1f}%")
        print()

## Summary and Conclusions

Based on the comprehensive analysis of the solar irradiance data, several key patterns emerge:

### Key Findings:

1. **Seasonal Patterns**: Clear seasonal variations are visible in all irradiance components, with typical peaks during summer months and lows during winter months.

2. **Daily Patterns**: If hourly data is available, typical daily patterns show peak irradiance around solar noon (12-1 PM).

3. **Correlations**: Strong correlations typically exist between:
   - DHI and CDHI (diffuse components)
   - DNI and CDNI (direct components)
   - Actual vs. clear sky values show the impact of weather conditions

4. **Clear Sky vs. Actual**: The difference between clear sky and actual measurements indicates the impact of clouds and atmospheric conditions on solar irradiance.

### Practical Implications:

- **Solar System Design**: Peak irradiance periods identify optimal installation orientation and capacity planning
- **Energy Forecasting**: Seasonal patterns help predict energy generation throughout the year
- **Weather Impact**: Deviations from clear sky values quantify weather effects on solar energy production

This analysis provides a solid foundation for understanding solar irradiance patterns and can inform solar energy system design and operation decisions.